# Pipeline clasificación de frescura de trucha por imágenes de la región ocular

Un solo notebook para las seis arquitecturas y los tres escenarios. Presenta dos modalidades tanto para colab como para servidor :D

## 1. Configuración de rutas de entrada y salida de datos + entorno + modo rapido + modelos

In [ ]:
# ============================ CONFIGURACIÓN GENERAL ============================
ENTORNO = "servidor"          # "colab" o "servidor"
MODO_RAPIDO = False         # True: verificación corta en Colab (1 modelo, 1 escenario, 2 épocas, 1 semilla)

# Rutas de cada escenario (carpetas con train / value / test y subcarpetas por día)
RUTAS = {
    "colab": {
        "base_resultados": "/content/drive/MyDrive/resultados_v2",
        "escenarios": {
            "E1": "/content/drive/MyDrive/prototipo/tesis_2026/pruebas_2026_1/truchas_ojos",
            "E2": "/content/drive/MyDrive/prototipo/tesis_2026/pruebas_2026_2/truchas_ojos",
            "E3": "/content/drive/MyDrive/prototipo/tesis_2026/pruebas_2026_3/truchas_ojos",
        },
    },
    "servidor": {
        "base_resultados": "/home/servidorx/tesis_2026/resultados_v2",
        "escenarios": {
            "E1": "/home/servidorx/tesis_2026/escenario1/pruebas_2026_1/truchas_ojos",
            "E2": "/home/servidorx/tesis_2026/escenario2/pruebas_2026_2/truchas_ojos",
            "E3": "/home/servidorx/tesis_2026/escenario3/pruebas_2026_3/truchas_ojos",
        },
    },
}
NOMBRE_ESCENARIO = {"E1": "Escenario 1 (5 días)", "E2": "Escenario 2 (días 1 y 9)", "E3": "Escenario 3 (días 1, 5 y 9)"}

# Nombres de las carpetas de partición tal como están en disco
CARPETA_TRAIN, CARPETA_VAL, CARPETA_TEST = "train", "value", "test"

# Arquitecturas evaluadas (objetivo específico 2)
MODELOS = ["densenet121", "efficientnet_b0", "inception_v3", "mobilenet_v2", "resnet50", "resnext50_32x4d"]
ESCENARIOS = ["E1", "E2", "E3"]

# Configuraciones de hiperparámetros (las tres de la tesis; ahora el tamaño de lote SÍ se aplica)
CONFIGS = {
    "P1": {"epocas": 15, "lote": 4,  "lr": 1e-3},
    "P2": {"epocas": 20, "lote": 8,  "lr": 5e-4},
    "P3": {"epocas": 25, "lote": 16, "lr": 1e-4},
}
OPTIMIZADOR = "adam"       # optimizador del estudio (objetivo específico 2)

# Repeticiones con semillas distintas (se reporta media ± desviación estándar)
SEMILLAS = [42, 123, 2026]

# Línea base del antecedente regional (ResNeXt50 + SGD). Ajustar a los valores de sus notebooks si se obtienen.
INCLUIR_LINEA_BASE = True
LINEA_BASE = {"modelo": "resnext50_32x4d", "optimizador": "sgd", "epocas": 30, "lote": 8, "lr": 1e-3, "momentum": 0.9}

# Otros
PESOS_PREENTRENADOS = True  # pesos de ImageNet (poner False solo para pruebas sin internet)
GUARDAR_PESOS = False       # guardar el .pt del mejor punto de control de cada corrida (ocupa espacio). IMPORTANTE SI QUIERES INFERIR UNA VEZ MAS!!!!
MEDIR_CPU = True            # medir también el tiempo de inferencia en CPU (equipo convencional)
NUM_WORKERS = 2 if ENTORNO == "colab" else 8
DISPOSITIVO = "cuda"        # en el servidor se elige la GPU con CUDA_VISIBLE_DEVICES

if MODO_RAPIDO:
    MODELOS, ESCENARIOS, SEMILLAS = ["mobilenet_v2"], ["E2"], [42]
    CONFIGS = {"P1": {**CONFIGS["P1"], "epocas": 2}}
    LINEA_BASE = {**LINEA_BASE, "epocas": 2}

## 2. Librerías y montaje de Drive

In [ ]:
import os, json, time, math, random, hashlib, datetime, platform, warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torchvision
from torchvision import datasets, models, transforms
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             confusion_matrix, classification_report)

warnings.filterwarnings("ignore", category=UserWarning)

if ENTORNO == "colab":
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except ImportError:
        print("No se está en Colab; se omite el montaje de Drive.")

RUTA_RESULTADOS = RUTAS[ENTORNO]["base_resultados"] + ("_rapido" if MODO_RAPIDO else "")
RUTAS_ESCENARIOS = RUTAS[ENTORNO]["escenarios"]
for sub in ["", "historiales", "predicciones", "figuras", "modelos", "resumen"]:
    os.makedirs(os.path.join(RUTA_RESULTADOS, sub), exist_ok=True)

DEVICE = torch.device(DISPOSITIVO if torch.cuda.is_available() else "cpu")
print(f"Python {platform.python_version()} | PyTorch {torch.__version__} | torchvision {torchvision.__version__}")
print(f"Dispositivo: {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else ""))
print(f"Resultados en: {RUTA_RESULTADOS}")

## 3. Reproducibilidad

In [ ]:
def fijar_semilla(semilla: int):
    """Fija todas las fuentes de aleatoriedad para que una corrida se pueda repetir."""
    random.seed(semilla)
    np.random.seed(semilla)
    torch.manual_seed(semilla)
    torch.cuda.manual_seed_all(semilla)
    os.environ["PYTHONHASHSEED"] = str(semilla)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def _semilla_worker(worker_id):
    s = torch.initial_seed() % 2**32
    np.random.seed(s)
    random.seed(s)

def guardar_entorno():
    """Registra versiones y hardware usados (para la sección de materiales de la tesis)."""
    info = {
        "fecha": datetime.datetime.now().isoformat(timespec="seconds"),
        "entorno": ENTORNO, "python": platform.python_version(), "sistema": platform.platform(),
        "pytorch": torch.__version__, "torchvision": torchvision.__version__,
        "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "n_gpu": torch.cuda.device_count(),
    }
    with open(os.path.join(RUTA_RESULTADOS, "entorno.json"), "w") as f:
        json.dump(info, f, indent=2)
    return info

print(guardar_entorno())

## 4. Verificación de la partición
Conteo de imágenes por escenario, partición, día y detección de
imágenes idénticas o casi idénticas entre particiones. Si aparecen pares entre `train` y `test`
(por ejemplo, dos fotos del mismo ojo), la exactitud en prueba queda inflada y hay que reubicarlos. Se corrige.

In [ ]:
EXT_IMG = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")

def listar_imagenes(ruta_escenario):
    filas = []
    for particion in [CARPETA_TRAIN, CARPETA_VAL, CARPETA_TEST]:
        ruta_p = os.path.join(ruta_escenario, particion)
        if not os.path.isdir(ruta_p):
            raise FileNotFoundError(f"No existe la carpeta {ruta_p}")
        for clase in sorted(os.listdir(ruta_p)):
            ruta_c = os.path.join(ruta_p, clase)
            if not os.path.isdir(ruta_c):
                continue
            for nombre in sorted(os.listdir(ruta_c)):
                if nombre.lower().endswith(EXT_IMG):
                    filas.append({"particion": particion, "clase": clase, "ruta": os.path.join(ruta_c, nombre)})
    return pd.DataFrame(filas)

def dhash_bits(ruta, tam=16):
    """Hash perceptual por diferencias (16x16 = 256 bits): imágenes casi iguales dan hashes cercanos."""
    img = Image.open(ruta).convert("L").resize((tam + 1, tam), Image.BILINEAR)
    a = np.asarray(img, dtype=np.int16)
    return (a[:, 1:] > a[:, :-1]).flatten()

def md5(ruta):
    with open(ruta, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

def verificar_escenario(clave, umbral_hamming=12):
    df = listar_imagenes(RUTAS_ESCENARIOS[clave])
    conteo = df.pivot_table(index="clase", columns="particion", values="ruta", aggfunc="count", fill_value=0)
    conteo = conteo.reindex(columns=[CARPETA_TRAIN, CARPETA_VAL, CARPETA_TEST], fill_value=0)
    conteo["total"] = conteo.sum(axis=1)
    conteo.loc["total"] = conteo.sum()

    # 1) Copias exactas: archivos de val/test idénticos byte a byte a alguno de train
    df["md5"] = df["ruta"].map(md5)
    md5_train = set(df.loc[df["particion"] == CARPETA_TRAIN, "md5"])
    eval_ = df[df["particion"] != CARPETA_TRAIN].reset_index(drop=True)
    train_ = df[df["particion"] == CARPETA_TRAIN].reset_index(drop=True)
    n_exactos = int(eval_["md5"].isin(md5_train).sum())

    # 2) Casi duplicados: distancia de Hamming entre hashes perceptuales (256 bits)
    b_train = np.stack([dhash_bits(r) for r in train_["ruta"]])
    b_eval = np.stack([dhash_bits(r) for r in eval_["ruta"]])
    dist = (b_eval[:, None, :] != b_train[None, :, :]).sum(axis=2)   # matriz eval x train
    i_ev, i_tr = np.where(dist <= umbral_hamming)
    sospechosos = pd.DataFrame({
        "eval": eval_.loc[i_ev, "ruta"].values, "particion_eval": eval_.loc[i_ev, "particion"].values,
        "clase_eval": eval_.loc[i_ev, "clase"].values, "train": train_.loc[i_tr, "ruta"].values,
        "clase_train": train_.loc[i_tr, "clase"].values, "hamming": dist[i_ev, i_tr],
    }).sort_values("hamming")

    print(f"\n=== {NOMBRE_ESCENARIO[clave]} ===")
    print(conteo.to_string())
    print(f"Imágenes de val/test copiadas exactamente de train: {n_exactos}")
    print(f"Pares casi idénticos (Hamming ≤ {umbral_hamming} de 256) entre train y val/test: {len(sospechosos)} "
          f"(afectan a {sospechosos['eval'].nunique() if len(sospechosos) else 0} imágenes de val/test)")
    print(f"Distancia mínima de cada imagen de val/test a train — mediana: {np.median(dist.min(1)):.0f} bits")
    conteo.to_csv(os.path.join(RUTA_RESULTADOS, "resumen", f"conteo_{clave}.csv"))
    if len(sospechosos):
        sospechosos.to_csv(os.path.join(RUTA_RESULTADOS, "resumen", f"posibles_fugas_{clave}.csv"), index=False)
        print("Revisar a ojo los pares del archivo posibles_fugas (los de menor distancia primero).")
    return conteo, sospechosos

VERIFICACION = {c: verificar_escenario(c) for c in ESCENARIOS}

## 5. Datos: transformaciones y cargadores
Entrenamiento con aumento leve; validación y prueba sin aumento.

In [ ]:
MEDIA, DE = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

def tamano_entrada(modelo):
    return 299 if modelo == "inception_v3" else 224

def transformaciones(modelo):
    t = tamano_entrada(modelo)
    entrenamiento = transforms.Compose([
        transforms.Resize((t, t)),
        transforms.RandomRotation(5),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(MEDIA, DE),
    ])
    evaluacion = transforms.Compose([
        transforms.Resize((t, t)),
        transforms.ToTensor(),
        transforms.Normalize(MEDIA, DE),
    ])
    return entrenamiento, evaluacion

def dia_de_clase(nombre_carpeta):
    """'1' -> 1, 'dia5' -> 5. Se usa para el error en días."""
    digitos = "".join(ch for ch in nombre_carpeta if ch.isdigit())
    return int(digitos) if digitos else None

def crear_cargadores(clave_escenario, modelo, lote, semilla):
    ruta = RUTAS_ESCENARIOS[clave_escenario]
    t_train, t_eval = transformaciones(modelo)
    ds_train = datasets.ImageFolder(os.path.join(ruta, CARPETA_TRAIN), transform=t_train)
    ds_val = datasets.ImageFolder(os.path.join(ruta, CARPETA_VAL), transform=t_eval)
    ds_test = datasets.ImageFolder(os.path.join(ruta, CARPETA_TEST), transform=t_eval)
    assert ds_train.classes == ds_val.classes == ds_test.classes, "Las clases no coinciden entre particiones"
    # Las carpetas '1','3','5','7','9' se ordenan bien; se ordena por día por si los nombres fueran distintos
    dias = [dia_de_clase(c) for c in ds_train.classes]
    assert dias == sorted(dias), f"Orden de clases inesperado: {ds_train.classes}"

    g = torch.Generator().manual_seed(semilla)
    comunes = dict(num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"), worker_init_fn=_semilla_worker)
    dl_train = torch.utils.data.DataLoader(ds_train, batch_size=lote, shuffle=True, generator=g, **comunes)
    dl_val = torch.utils.data.DataLoader(ds_val, batch_size=32, shuffle=False, **comunes)
    dl_test = torch.utils.data.DataLoader(ds_test, batch_size=32, shuffle=False, **comunes)
    return dl_train, dl_val, dl_test, ds_train.classes, dias, [p for p, _ in ds_test.samples]

## 6. Arquitecturas
Pesos de ImageNet y solo se reemplaza la última capa lineal. La salida son *logits* (sin softmax).

In [ ]:
def crear_modelo(nombre, n_clases, preentrenado=True):
    W = lambda enum: enum.DEFAULT if preentrenado else None
    if nombre == "densenet121":
        m = models.densenet121(weights=W(models.DenseNet121_Weights))
        m.classifier = nn.Linear(m.classifier.in_features, n_clases)
    elif nombre == "efficientnet_b0":
        m = models.efficientnet_b0(weights=W(models.EfficientNet_B0_Weights))
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, n_clases)  # conserva el Dropout(0.2)
    elif nombre == "inception_v3":
        m = models.inception_v3(weights=W(models.Inception_V3_Weights), aux_logits=True,
                                init_weights=not preentrenado)
        m.fc = nn.Linear(m.fc.in_features, n_clases)
        m.AuxLogits.fc = nn.Linear(m.AuxLogits.fc.in_features, n_clases)
    elif nombre == "mobilenet_v2":
        m = models.mobilenet_v2(weights=W(models.MobileNet_V2_Weights))
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, n_clases)  # conserva el Dropout(0.2)
    elif nombre == "resnet50":
        m = models.resnet50(weights=W(models.ResNet50_Weights))
        m.fc = nn.Linear(m.fc.in_features, n_clases)
    elif nombre == "resnext50_32x4d":
        m = models.resnext50_32x4d(weights=W(models.ResNeXt50_32X4D_Weights))
        m.fc = nn.Linear(m.fc.in_features, n_clases)
    else:
        raise ValueError(f"Modelo no reconocido: {nombre}")
    return m

def contar_parametros(m):
    return sum(p.numel() for p in m.parameters())

## 7. Entrenamiento con selección del mejor punto de control por validación

In [ ]:
PESO_AUX_INCEPTION = 0.4

def salida_y_perdida(modelo, x, y, criterio):
    """Devuelve (logits principales, pérdida). En InceptionV3 suma la salida auxiliar durante el entrenamiento."""
    out = modelo(x)
    if isinstance(out, torchvision.models.inception.InceptionOutputs):
        perdida = criterio(out.logits, y) + PESO_AUX_INCEPTION * criterio(out.aux_logits, y)
        return out.logits, perdida
    return out, criterio(out, y)

@torch.no_grad()
def evaluar(modelo, cargador, criterio):
    modelo.eval()
    perdida, y_true, y_pred, probs = 0.0, [], [], []
    for x, y in cargador:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        logits = modelo(x)
        perdida += criterio(logits, y).item() * y.size(0)
        p = torch.softmax(logits, dim=1)
        probs.append(p.cpu().numpy())
        y_pred.append(p.argmax(1).cpu().numpy())
        y_true.append(y.cpu().numpy())
    y_true, y_pred = np.concatenate(y_true), np.concatenate(y_pred)
    return perdida / len(y_true), y_true, y_pred, np.concatenate(probs)

def crear_optimizador(nombre, parametros, lr, momentum=0.9):
    if nombre == "adam":
        return torch.optim.Adam(parametros, lr=lr)
    if nombre == "sgd":
        return torch.optim.SGD(parametros, lr=lr, momentum=momentum)
    raise ValueError(nombre)

def entrenar(modelo, dl_train, dl_val, epocas, lr, optimizador="adam", momentum=0.9):
    criterio = nn.CrossEntropyLoss()
    opt = crear_optimizador(optimizador, modelo.parameters(), lr, momentum)
    historial, mejor = [], {"val_acc": -1.0, "val_loss": float("inf"), "epoca": 0, "estado": None}
    t0 = time.time()
    for ep in range(1, epocas + 1):
        modelo.train()
        suma_loss, aciertos, n = 0.0, 0, 0
        for x, y in dl_train:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            logits, loss = salida_y_perdida(modelo, x, y, criterio)
            loss.backward()
            opt.step()
            suma_loss += loss.item() * y.size(0)
            aciertos += (logits.argmax(1) == y).sum().item()
            n += y.size(0)
        val_loss, yv, pv, _ = evaluar(modelo, dl_val, criterio)
        val_acc = 100 * (yv == pv).mean()
        fila = {"epoca": ep, "train_loss": suma_loss / n, "train_acc": 100 * aciertos / n,
                "val_loss": val_loss, "val_acc": val_acc}
        historial.append(fila)
        # Mejor punto de control: mayor exactitud de validación; empate -> menor pérdida de validación
        if (val_acc > mejor["val_acc"]) or (val_acc == mejor["val_acc"] and val_loss < mejor["val_loss"]):
            mejor = {"val_acc": val_acc, "val_loss": val_loss, "epoca": ep,
                     "estado": {k: v.detach().cpu().clone() for k, v in modelo.state_dict().items()}}
        print(f"  Época {ep:>2}/{epocas} | train {fila['train_acc']:6.2f}% (loss {fila['train_loss']:.3f})"
              f" | val {val_acc:6.2f}% (loss {val_loss:.3f})")
    modelo.load_state_dict(mejor["estado"])
    return pd.DataFrame(historial), mejor, time.time() - t0

## 8. Métricas de evaluación

In [ ]:
def ic_wilson(aciertos, n, z=1.96):
    """Intervalo de confianza del 95 % de Wilson para una proporción (exactitud)."""
    if n == 0:
        return (float("nan"), float("nan"))
    p = aciertos / n
    d = 1 + z**2 / n
    c = (p + z**2 / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / d
    return 100 * (c - h), 100 * (c + h)

def mcnemar_exacto(correcto_a, correcto_b):
    """Prueba exacta de McNemar (bilateral) sobre los mismos ejemplos de prueba."""
    a, b = np.asarray(correcto_a, bool), np.asarray(correcto_b, bool)
    n01, n10 = int((a & ~b).sum()), int((~a & b).sum())
    n = n01 + n10
    if n == 0:
        return n01, n10, 1.0
    k = min(n01, n10)
    p = sum(math.comb(n, i) for i in range(k + 1)) / 2**n
    return n01, n10, min(1.0, 2 * p)

@torch.no_grad()
def tiempo_inferencia_ms(modelo, dl_test, dispositivo, max_img=50):
    """Milisegundos por imagen (lote de 1), tras un calentamiento."""
    modelo = modelo.to(dispositivo).eval()
    imgs = []
    for x, _ in dl_test:
        imgs.extend(x.split(1))
        if len(imgs) >= max_img:
            break
    imgs = imgs[:max_img]
    for x in imgs[:5]:
        modelo(x.to(dispositivo))
    if dispositivo.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for x in imgs:
        modelo(x.to(dispositivo))
    if dispositivo.type == "cuda":
        torch.cuda.synchronize()
    return 1000 * (time.perf_counter() - t0) / len(imgs)

def metricas(y_true, y_pred, dias):
    dias = np.asarray(dias)
    n, ac = len(y_true), int((y_true == y_pred).sum())
    lo, hi = ic_wilson(ac, n)
    return {
        "test_acc": 100 * ac / n, "test_acc_ic_inf": lo, "test_acc_ic_sup": hi, "n_test": n,
        "test_f1_macro": 100 * f1_score(y_true, y_pred, average="macro", zero_division=0),
        "test_precision_macro": 100 * precision_score(y_true, y_pred, average="macro", zero_division=0),
        "test_recall_macro": 100 * recall_score(y_true, y_pred, average="macro", zero_division=0),
        "test_mae_dias": float(np.abs(dias[y_true] - dias[y_pred]).mean()),
    }

## 9. Figuras por corrida

In [ ]:
def figura_curvas(hist, titulo, ruta):
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
    ax[0].plot(hist["epoca"], hist["train_loss"], label="Entrenamiento")
    ax[0].plot(hist["epoca"], hist["val_loss"], label="Validación")
    ax[0].set(xlabel="Época", ylabel="Pérdida", title="Pérdida")
    ax[1].plot(hist["epoca"], hist["train_acc"], label="Entrenamiento")
    ax[1].plot(hist["epoca"], hist["val_acc"], label="Validación")
    ax[1].set(xlabel="Época", ylabel="Exactitud (%)", title="Exactitud", ylim=(0, 100))
    for a in ax:
        a.grid(alpha=.3)
        a.legend()
    fig.suptitle(titulo, fontsize=10)
    fig.tight_layout()
    fig.savefig(ruta, dpi=150)
    plt.close(fig)

def figura_confusion(cm, etiquetas, titulo, ruta):
    fig, ax = plt.subplots(figsize=(4.2, 3.8))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(etiquetas)), etiquetas)
    ax.set_yticks(range(len(etiquetas)), etiquetas)
    ax.set(xlabel="Día predicho", ylabel="Día real", title=titulo)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    fig.colorbar(im, ax=ax, fraction=.046)
    fig.tight_layout()
    fig.savefig(ruta, dpi=150)
    plt.close(fig)

## 10. Cada corrida completa equivale a entrenar → evaluar → guardar
Cada corrida queda registrada en `corridas.csv`. Si Colab queda desconectado, al volver a
ejecutar se omiten las corridas ya terminadas :D asi que no te preocupes si llegas al maximo uso por dia.

In [ ]:
ARCHIVO_CORRIDAS = os.path.join(RUTA_RESULTADOS, "corridas.csv")

def corridas_hechas():
    if os.path.exists(ARCHIVO_CORRIDAS):
        return set(pd.read_csv(ARCHIVO_CORRIDAS)["run_id"])
    return set()

def registrar(fila):
    df = pd.DataFrame([fila])
    df.to_csv(ARCHIVO_CORRIDAS, mode="a", header=not os.path.exists(ARCHIVO_CORRIDAS), index=False)

def ejecutar_corrida(modelo_nombre, esc, config_id, cfg, semilla, optimizador="adam", momentum=0.9, grupo="estudio"):
    run_id = f"{grupo}_{modelo_nombre}_{esc}_{config_id}_s{semilla}"
    if run_id in corridas_hechas():
        print(f"[omitida] {run_id} ya existe")
        return
    print(f"\n>>> {run_id}  ({NOMBRE_ESCENARIO[esc]}, {cfg['epocas']} épocas, lote {cfg['lote']}, lr {cfg['lr']}, {optimizador})")
    fijar_semilla(semilla)
    dl_tr, dl_va, dl_te, clases, dias, rutas_test = crear_cargadores(esc, modelo_nombre, cfg["lote"], semilla)
    modelo = crear_modelo(modelo_nombre, len(clases), PESOS_PREENTRENADOS).to(DEVICE)

    hist, mejor, t_entr = entrenar(modelo, dl_tr, dl_va, cfg["epocas"], cfg["lr"], optimizador, momentum)
    _, yt, yp, probs = evaluar(modelo, dl_te, nn.CrossEntropyLoss())
    m = metricas(yt, yp, dias)
    cm = confusion_matrix(yt, yp, labels=list(range(len(clases))))

    t_gpu = tiempo_inferencia_ms(modelo, dl_te, DEVICE) if DEVICE.type == "cuda" else float("nan")
    t_cpu = tiempo_inferencia_ms(modelo, dl_te, torch.device("cpu")) if MEDIR_CPU else float("nan")
    modelo.to(DEVICE)

    fila_mejor = hist.loc[hist["epoca"] == mejor["epoca"]].iloc[0]
    fila = {
        "run_id": run_id, "grupo": grupo, "modelo": modelo_nombre, "escenario": esc, "config": config_id,
        "semilla": semilla, "optimizador": optimizador, "epocas": cfg["epocas"], "lote": cfg["lote"], "lr": cfg["lr"],
        "mejor_epoca": mejor["epoca"], "train_acc": fila_mejor["train_acc"], "val_acc": mejor["val_acc"],
        "val_loss": mejor["val_loss"], **m,
        "parametros_M": contar_parametros(modelo) / 1e6, "t_entrenamiento_s": t_entr,
        "t_inferencia_gpu_ms": t_gpu, "t_inferencia_cpu_ms": t_cpu,
        "fecha": datetime.datetime.now().isoformat(timespec="seconds"),
    }
    hist.to_csv(os.path.join(RUTA_RESULTADOS, "historiales", f"{run_id}.csv"), index=False)
    pred = pd.DataFrame({"ruta": rutas_test, "y_true": yt, "y_pred": yp,
                         "dia_real": np.array(dias)[yt], "dia_pred": np.array(dias)[yp]})
    for k, c in enumerate(clases):
        pred[f"prob_{c}"] = probs[:, k]
    pred.to_csv(os.path.join(RUTA_RESULTADOS, "predicciones", f"{run_id}.csv"), index=False)
    etiquetas = [str(d) for d in dias]
    figura_curvas(hist, run_id, os.path.join(RUTA_RESULTADOS, "figuras", f"{run_id}_curvas.png"))
    figura_confusion(cm, etiquetas, f"{modelo_nombre} · {esc} · {config_id}",
                     os.path.join(RUTA_RESULTADOS, "figuras", f"{run_id}_confusion.png"))
    if GUARDAR_PESOS:
        torch.save(modelo.state_dict(), os.path.join(RUTA_RESULTADOS, "modelos", f"{run_id}.pt"))
    registrar(fila)
    print(f"<<< prueba {m['test_acc']:.2f}% (IC95 {m['test_acc_ic_inf']:.1f}–{m['test_acc_ic_sup']:.1f}) | "
          f"F1 {m['test_f1_macro']:.2f}% | error medio {m['test_mae_dias']:.2f} días | mejor época {mejor['epoca']}")
    del modelo
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

## 11. Prueba rápida de las arquitecturas
Pasa un lote por cada modelo para detectar errores de forma antes de entrenar.

In [ ]:
def prueba_humo():
    for nombre in MODELOS + ([LINEA_BASE["modelo"]] if INCLUIR_LINEA_BASE else []):
        t = tamano_entrada(nombre)
        m = crear_modelo(nombre, 3, preentrenado=False).to(DEVICE)
        m.train()
        x, y = torch.randn(2, 3, t, t, device=DEVICE), torch.tensor([0, 2], device=DEVICE)
        logits, loss = salida_y_perdida(m, x, y, nn.CrossEntropyLoss())
        loss.backward()
        m.eval()
        with torch.no_grad():
            out = m(x)
        assert out.shape == (2, 3), out.shape
        print(f"OK {nombre:<16} entrada {t}px | salida {tuple(out.shape)} | {contar_parametros(m)/1e6:.2f} M parámetros")
        del m

prueba_humo()

## 12. Ejecución del estudio
6 arquitecturas × 3 escenarios × 3 configuraciones × semillas. En el servidor se puede los repartir modelos
entre las dos GPU (ver el script `pipeline_tesis.py`). Sin embargo consideré que requiero exactitud y no velocidad.

In [ ]:
def ejecutar_estudio():
    for esc in ESCENARIOS:
        for nombre in MODELOS:
            for cid, cfg in CONFIGS.items():
                for s in SEMILLAS:
                    ejecutar_corrida(nombre, esc, cid, cfg, s, optimizador=OPTIMIZADOR, grupo="estudio")

def ejecutar_linea_base():
    if not INCLUIR_LINEA_BASE:
        return
    lb = LINEA_BASE
    cfg = {"epocas": lb["epocas"], "lote": lb["lote"], "lr": lb["lr"]}
    for esc in ESCENARIOS:
        for s in SEMILLAS:
            ejecutar_corrida(lb["modelo"], esc, "LB", cfg, s, optimizador=lb["optimizador"],
                             momentum=lb["momentum"], grupo="lineabase")

ejecutar_estudio()
ejecutar_linea_base()

## 13. Resumen mis resultados
- **Tabla A**: media ± DE por modelo, escenario y configuración (todas las corridas).
- **Tabla B**: para cada modelo y escenario se elige la configuración con
  mayor exactitud media de validación y se reporta su exactitud de prueba.
- **Hipótesis 3**: cuántas arquitecturas alcanzan ≥ 85 % en los escenarios 2 y 3.
- **Optimización frente a la línea base**: diferencia y prueba de McNemar por semilla.

In [ ]:
def ms(serie):
    return f"{serie.mean():.2f} ± {serie.std(ddof=1):.2f}" if len(serie) > 1 else f"{serie.mean():.2f}"

def resumir():
    df = pd.read_csv(ARCHIVO_CORRIDAS)
    est = df[df["grupo"] == "estudio"]
    metr = ["train_acc", "val_acc", "test_acc", "test_f1_macro", "test_mae_dias"]

    # Tabla A
    tabla_a = est.groupby(["modelo", "escenario", "config"]).agg(
        n=("semilla", "count"), **{f"{c}_media": (c, "mean") for c in metr},
        **{f"{c}_de": (c, "std") for c in metr}).reset_index()
    tabla_a.to_csv(os.path.join(RUTA_RESULTADOS, "resumen", "tabla_A_todas_las_configuraciones.csv"), index=False)

    # Tabla B: configuración elegida por validación
    idx = tabla_a.groupby(["modelo", "escenario"])["val_acc_media"].idxmax()
    elegidas = tabla_a.loc[idx, ["modelo", "escenario", "config"]]
    filas = []
    for _, r in elegidas.iterrows():
        sub = est[(est.modelo == r.modelo) & (est.escenario == r.escenario) & (est.config == r.config)]
        filas.append({"modelo": r.modelo, "escenario": r.escenario, "config_elegida": r.config,
                      "val_acc": ms(sub.val_acc), "val_acc_media": sub.val_acc.mean(),
                      "test_acc": ms(sub.test_acc), "test_acc_media": sub.test_acc.mean(),
                      "test_f1_macro": ms(sub.test_f1_macro), "error_dias": ms(sub.test_mae_dias),
                      "parametros_M": round(sub.parametros_M.iloc[0], 2),
                      "t_cpu_ms": round(sub.t_inferencia_cpu_ms.mean(), 1)})
    # Orden por validación: el "mejor modelo" de cada escenario se elige sin mirar la prueba
    tabla_b = pd.DataFrame(filas).sort_values(["escenario", "val_acc_media"], ascending=[True, False])
    tabla_b.to_csv(os.path.join(RUTA_RESULTADOS, "resumen", "tabla_B_config_por_validacion.csv"), index=False)
    print("\nTabla B — configuración elegida por validación (exactitud de prueba, media ± DE)")
    print(tabla_b.drop(columns=["test_acc_media", "val_acc_media"]).to_string(index=False))

    # Hipótesis 3
    for esc in ["E2", "E3"]:
        t = tabla_b[tabla_b.escenario == esc]
        if len(t):
            k = int((t.test_acc_media >= 85).sum())
            print(f"H3 · {NOMBRE_ESCENARIO[esc]}: {k} de {len(t)} arquitecturas con exactitud media ≥ 85 %")

    # Optimización frente a la línea base (McNemar por semilla, sobre las mismas imágenes de prueba)
    lb = df[df["grupo"] == "lineabase"]
    if len(lb):
        print("\nComparación con la línea base (ResNeXt50 + SGD)")
        filas_c = []
        for esc in sorted(lb.escenario.unique()):
            tb = tabla_b[tabla_b.escenario == esc]
            if not len(tb):
                continue
            mejor = tb.iloc[0]  # mejor modelo del escenario según validación
            for s in sorted(lb[lb.escenario == esc].semilla):
                id_m = f"estudio_{mejor.modelo}_{esc}_{mejor.config_elegida}_s{s}"
                id_b = f"lineabase_{LINEA_BASE['modelo']}_{esc}_LB_s{s}"
                fm = os.path.join(RUTA_RESULTADOS, "predicciones", f"{id_m}.csv")
                fb = os.path.join(RUTA_RESULTADOS, "predicciones", f"{id_b}.csv")
                if not (os.path.exists(fm) and os.path.exists(fb)):
                    continue
                pm, pb = pd.read_csv(fm), pd.read_csv(fb)
                assert (pm.ruta == pb.ruta).all()
                n01, n10, p = mcnemar_exacto(pm.y_true == pm.y_pred, pb.y_true == pb.y_pred)
                filas_c.append({"escenario": esc, "semilla": s, "modelo": mejor.modelo,
                                "acc_modelo": 100 * (pm.y_true == pm.y_pred).mean(),
                                "acc_linea_base": 100 * (pb.y_true == pb.y_pred).mean(),
                                "solo_modelo_acierta": n01, "solo_base_acierta": n10, "p_mcnemar": p})
        comp = pd.DataFrame(filas_c)
        if len(comp):
            comp.to_csv(os.path.join(RUTA_RESULTADOS, "resumen", "comparacion_linea_base.csv"), index=False)
            print(comp.round(4).to_string(index=False))

    # Figura: exactitud de prueba (configuración elegida) con barras de error
    fig, ax = plt.subplots(figsize=(8, 4.5))
    escs = [e for e in ["E1", "E2", "E3"] if e in set(tabla_b.escenario)]
    mods = sorted(tabla_b.modelo.unique())
    ancho = 0.8 / max(1, len(escs))
    for k, esc in enumerate(escs):
        medias, des = [], []
        for mo in mods:
            r = elegidas[(elegidas.modelo == mo) & (elegidas.escenario == esc)]
            sub = est[(est.modelo == mo) & (est.escenario == esc) & (est.config == r.config.iloc[0])] if len(r) else est.iloc[0:0]
            medias.append(sub.test_acc.mean() if len(sub) else np.nan)
            des.append(sub.test_acc.std(ddof=1) if len(sub) > 1 else 0)
        ax.barh(np.arange(len(mods)) + k * ancho, medias, ancho, xerr=des, label=NOMBRE_ESCENARIO[esc], capsize=3)
    ax.set_yticks(np.arange(len(mods)) + ancho * (len(escs) - 1) / 2, mods)
    ax.axvline(85, ls="--", c="gray", lw=1)
    ax.set(xlabel="Exactitud en prueba (%) — configuración elegida por validación", xlim=(0, 100))
    ax.legend(loc="lower right", fontsize=8)
    fig.tight_layout()
    fig.savefig(os.path.join(RUTA_RESULTADOS, "resumen", "figura_exactitud_por_escenario.png"), dpi=200)
    plt.show()
    return tabla_a, tabla_b

TABLA_A, TABLA_B = resumir()